In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Load Dataset and Initial Inspection ---
# Replace 'path/to/your/sirio_libanes_covid_data.csv' with the actual path to your dataset
try:
    # Corrected: Use pd.read_excel for .xlsx files
    df = pd.read_excel('/content/Kaggle_Sirio_Libanes_ICU_Prediction.xlsx')
except FileNotFoundError:
    print("Error: Dataset not found. Please update 'path/to/your/sirio_libanes_covid_data.csv'")
    exit()

print("Original dataset shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())

# Identify target variable (adjust if your dataset's column name is different)
TARGET_COLUMN = 'ICU' # Common target for ICU admission, adjust if needed

if TARGET_COLUMN not in df.columns:
    print(f"Error: Target column '{TARGET_COLUMN}' not found in dataset. Please specify the correct target column.")
    exit()

# Separate features (X) and target (y)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print(f"\nTarget variable distribution:\n{y.value_counts(normalize=True)}")

# --- 2. Data Cleaning and Preprocessing Pipeline ---

# Step 1: Detect missing values and visualize (optional, but good for understanding)
print("\nMissing values before cleaning:")
missing_values = X.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)

# Visualizing missing values (requires matplotlib and seaborn)
# plt.figure(figsize=(12, 6))
# sns.heatmap(X.isnull(), cbar=False, cmap='viridis')
# plt.title("Missing Values Heatmap (Before Cleaning)")
# plt.show()


# Store original column names for later comparison
original_cols = X.columns.tolist()

# Define numerical and categorical features (adjust based on your dataset)
# It's crucial to inspect your dataset's columns and types
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# Example: If 'AGE_ABOVE65' is numerical but should be treated as a binary categorical, move it
# if 'AGE_ABOVE65' in numerical_cols:
#     numerical_cols.remove('AGE_ABOVE65')
#     categorical_cols.append('AGE_ABOVE65')

# Identify potential 'WINDOW' columns for specific handling if they are categorical strings
window_cols = [col for col in categorical_cols if 'WINDOW' in col.upper()]
if window_cols:
    print(f"Detected window columns: {window_cols}. Assuming they might be handled as ordinal later or one-hot encoded.")
    # For this example, let's assume they are handled by OneHotEncoder if left in categorical_cols

# Define age category column (adjust if name is different)
AGE_CATEGORY_COL = [col for col in X.columns if 'AGE_PERCENTIL' in col.upper()] # Common name
if AGE_CATEGORY_COL:
    AGE_CATEGORY_COL = AGE_CATEGORY_COL[0]
    if AGE_CATEGORY_COL in categorical_cols:
        categorical_cols.remove(AGE_CATEGORY_COL)
    # Ensure it's in a specific list if you have a custom ordinal mapping
    ordinal_cols = [AGE_CATEGORY_COL]
else:
    ordinal_cols = []
    print("Warning: Age category column not found. Skipping ordinal encoding for age.")

print(f"\nIdentified Numerical Columns: {numerical_cols}")
print(f"Identified Categorical Columns: {categorical_cols}")
print(f"Identified Ordinal Columns (Age): {ordinal_cols}")


# Step 2: Drop features with >50% missing values
initial_feature_count = X.shape[1]
threshold = 0.5 * X.shape[0]
X = X.dropna(axis=1, thresh=threshold)
print(f"\nDropped columns with >50% missing values. Remaining features: {X.shape[1]} (Removed: {initial_feature_count - X.shape[1]})")

# Update numerical and categorical lists after dropping columns
numerical_cols = [col for col in numerical_cols if col in X.columns]
categorical_cols = [col for col in categorical_cols if col in X.columns]
ordinal_cols = [col for col in ordinal_cols if col in X.columns]


# Imputation for remaining missing values
# For numerical features: median for skewed (e.g., lab results often are), mean otherwise.
# A simpler approach for general use below is median for all numericals.
# For categorical features: mode.
for col in numerical_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].median())
        # print(f"Imputed numerical column '{col}' with median.")
for col in categorical_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])
        # print(f"Imputed categorical column '{col}' with mode.")
for col in ordinal_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])
        # print(f"Imputed ordinal column '{col}' with mode.")

print("\nMissing values after imputation:")
print(X.isnull().sum().sum()) # Should be 0

# Step 3: Detect and cap outliers (O2 saturation and Creatinine)
# Need to identify these columns from your dataset. Example names:
O2_SAT_COL = [col for col in numerical_cols if 'OXYGEN_SATURATION' in col.upper() or 'SPO2' in col.upper()]
CREATININE_COL = [col for col in numerical_cols if 'CREATININE' in col.upper()]

def cap_outliers_iqr(df_col):
    Q1 = df_col.quantile(0.25)
    Q3 = df_col.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df_col.clip(lower=lower_bound, upper=upper_bound)

print("\nCapping outliers (if columns found):")
if O2_SAT_COL:
    for col in O2_SAT_COL:
        if col in X.columns:
            print(f"Capping outliers for {col}")
            X[col] = cap_outliers_iqr(X[col])
if CREATININE_COL:
    for col in CREATININE_COL:
        if col in X.columns:
            print(f"Capping outliers for {col}")
            X[col] = cap_outliers_iqr(X[col])

# Step 4: Encode age category (ordinal), One-Hot encode admission type/other categorical
# Ordinal encoding for age percentiles (assuming they are strings like '0-20', '20-40', etc.)
# You might need to define a specific order based on your data's age categories
if ordinal_cols:
    age_categories_order = ['0-2', '2-4', '4-6', '6-12', 'ABOVE_12', 'ABOVE-12'] # Adjust based on your dataset's actual age percentiles e.g. '0-20', '20-40', etc. or WINDOWs
    age_encoder = OrdinalEncoder(categories=[age_categories_order], handle_unknown='use_encoded_value', unknown_value=-1)
    for col in ordinal_cols:
        if col in X.columns : # Check again if column exists after previous steps
             # Ensure the column is of string type before encoding
            X[col] = X[col].astype(str)
            X[[col]] = age_encoder.fit_transform(X[[col]])
    print(f"Ordinal encoded: {ordinal_cols}")
    numerical_cols.extend(ordinal_cols) # Add to numerical for scaling later

# One-Hot encode remaining categorical features
# Identify categorical columns that are not ordinal or already handled
one_hot_cols = [col for col in categorical_cols if col not in ordinal_cols]

if one_hot_cols:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_features = encoder.fit_transform(X[one_hot_cols])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(one_hot_cols), index=X.index)
    X = pd.concat([X.drop(columns=one_hot_cols), encoded_df], axis=1)
    print(f"One-Hot encoded: {one_hot_cols}")

# Update numerical columns after encoding
numerical_cols = [col for col in numerical_cols if col not in one_hot_cols] # Remove one-hot encoded originals
# Add newly created one-hot encoded columns to numerical for scaling if needed
numerical_cols.extend(encoder.get_feature_names_out(one_hot_cols).tolist() if one_hot_cols else [])


# Step 5: Apply Z-score to continuous lab results, Min-Max to count features
# This step requires careful identification of 'lab results' vs. 'count features'.
# For simplicity, we'll apply StandardScaler (Z-score) to all numerical features.
# You would refine this by creating specific lists for lab_results_cols and count_features_cols.

print("\nApplying scaling...")
scaler_zscore = StandardScaler()
X[numerical_cols] = scaler_zscore.fit_transform(X[numerical_cols])
print("Applied Z-score scaling to numerical features.")

# Example of Min-Max for hypothetical count features (if you had them)
# count_features_cols = ['NUM_COMORBIDITIES', 'NUM_SYMPTOMS']
# scaler_minmax = MinMaxScaler()
# X[count_features_cols] = scaler_minmax.fit_transform(X[count_features_cols])
# print("Applied Min-Max scaling to count features.")


# Step 6: Engineer new feature: respiratory_risk_score
# You'll need to define the column names for SpO2 and respiration_rate from your dataset
SPO2_COL = [col for col in X.columns if 'OXYGEN_SATURATION' in col.upper() or 'SPO2' in col.upper()][0] if [col for col in X.columns if 'OXYGEN_SATURATION' in col.upper() or 'SPO2' in col.upper()] else None
RESP_RATE_COL = [col for col in X.columns if 'RESPIRATION_RATE' in col.upper()][0] if [col for col in X.columns if 'RESPIRATION_RATE' in col.upper()] else None

if SPO2_COL and RESP_RATE_COL:
    # This is a simplified example; actual medical risk scores are more complex.
    # A lower SpO2 and higher respiration rate indicate higher risk.
    # Assuming SpO2 is already scaled (Z-score is fine).
    # You might want to inverse SpO2 or create bins for a more clinical score.
    X['respiratory_risk_score'] = (X[RESP_RATE_COL] - X[SPO2_COL]) # Simple combination
    print(f"Engineered 'respiratory_risk_score' using {SPO2_COL} and {RESP_RATE_COL}.")
else:
    print("Warning: Could not find SpO2 or Respiration Rate columns for feature engineering.")


# --- 7. Train KNN before and after pipeline, report accuracy lift percentage ---

# 7.1. Prepare 'before' dataset (only dropped features, basic imputation, no scaling/encoding specific to pipeline)
# For a fair 'before' comparison, we should use a dataset that has minimal processing.
# Let's re-load the original data for a baseline comparison to show the *impact of the whole pipeline*.

# Corrected: Use pd.read_excel for .xlsx files
df_before = pd.read_excel('/content/Kaggle_Sirio_Libanes_ICU_Prediction.xlsx')
X_before = df_before.drop(columns=[TARGET_COLUMN])
y_before = df_before[TARGET_COLUMN]

# Step 1 (before): Drop features with >50% missing values
initial_feature_count_before = X_before.shape[1]
threshold_before = 0.5 * X_before.shape[0]
X_before = X_before.dropna(axis=1, thresh=threshold_before)

# Simplify imputation for baseline: use median for all numerical, mode for all categorical
numerical_cols_before = X_before.select_dtypes(include=np.number).columns.tolist()
categorical_cols_before = X_before.select_dtypes(exclude=np.number).columns.tolist()

for col in numerical_cols_before:
    if X_before[col].isnull().sum() > 0:
        X_before[col] = X_before[col].fillna(X_before[col].median())
for col in categorical_cols_before:
    if X_before[col].isnull().sum() > 0:
        X_before[col] = X_before[col].fillna(X_before[col].mode()[0])

# One-Hot encode all categorical for baseline, but no special ordinal or specific scaling
encoder_before = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
if categorical_cols_before:
    encoded_features_before = encoder_before.fit_transform(X_before[categorical_cols_before])
    encoded_df_before = pd.DataFrame(encoded_features_before,
                                     columns=encoder_before.get_feature_names_out(categorical_cols_before),
                                     index=X_before.index)
    X_before = pd.concat([X_before.drop(columns=categorical_cols_before), encoded_df_before], axis=1)

# Align columns for 'before' and 'after' if needed, though for KNN it often implies re-training.
# For simplicity, we'll train KNN on X_before and X_after separately using their own data.

# Split data for 'before' pipeline
X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X_before, y_before, test_size=0.2, random_state=42, stratify=y_before
)

# Train KNN (before pipeline)
knn_before = KNeighborsClassifier(n_neighbors=5) # n_neighbors is a hyperparameter to tune
knn_before.fit(X_train_before, y_train_before)
y_pred_before = knn_before.predict(X_test_before)
accuracy_before = accuracy_score(y_test_before, y_pred_before)
print(f"\nAccuracy BEFORE pipeline: {accuracy_before:.4f}")


# Split data for 'after' pipeline
X_train_after, X_test_after, y_train_after, y_test_after = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train KNN (after pipeline)
knn_after = KNeighborsClassifier(n_neighbors=5) # n_neighbors is a hyperparameter to tune
knn_after.fit(X_train_after, y_train_after)
y_pred_after = knn_after.predict(X_test_after)
accuracy_after = accuracy_score(y_test_after, y_pred_after)
print(f"Accuracy AFTER pipeline: {accuracy_after:.4f}")
print(f"\nClassification Report AFTER pipeline:\n{classification_report(y_test_after, y_pred_after)}")


# Report accuracy lift percentage
if accuracy_before > 0:
    lift_percentage = ((accuracy_after - accuracy_before) / accuracy_before) * 100
    print(f"\nAccuracy lift due to pipeline: {lift_percentage:.2f}%")
else:
    print("\nCannot calculate accuracy lift as 'before' accuracy is zero.")

# --- Further steps to achieve 90%+ accuracy (not in this script, but for your consideration) ---
# 1. Hyperparameter Tuning: Use GridSearchCV or RandomizedSearchCV for KNN (or other models).
#    E.g., for KNN: search for optimal 'n_neighbors'.
# 2. Try More Advanced Models: XGBoost, Random Forest, SVM. XGBoost is often very powerful for tabular data.
# 3. More Sophisticated Imputation: MICE (IterativeImputer).
# 4. Feature Selection: Recursive Feature Elimination (RFE), SelectKBest.
# 5. Handle Class Imbalance: SMOTE (if one class is much rarer than the other).
# 6. Ensemble Methods: Bagging, Boosting.
# 7. Domain-Specific Feature Engineering: Work closely with medical professionals if possible to create more meaningful features.

Original dataset shape: (1925, 231)

First 5 rows of the dataset:
   PATIENT_VISIT_IDENTIFIER  AGE_ABOVE65 AGE_PERCENTIL  GENDER  \
0                         0            1          60th       0   
1                         0            1          60th       0   
2                         0            1          60th       0   
3                         0            1          60th       0   
4                         0            1          60th       0   

   DISEASE GROUPING 1  DISEASE GROUPING 2  DISEASE GROUPING 3  \
0                 0.0                 0.0                 0.0   
1                 0.0                 0.0                 0.0   
2                 0.0                 0.0                 0.0   
3                 0.0                 0.0                 0.0   
4                 0.0                 0.0                 0.0   

   DISEASE GROUPING 4  DISEASE GROUPING 5  DISEASE GROUPING 6  ...  \
0                 0.0                 1.0                 1.0  ...   
1     